In [ ]:
import streamlit as st
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# -------------------- PAGE CONFIG --------------------
st.set_page_config(page_title="Genome Intelligence System", layout="wide")

st.title("Genome Intelligence System")
st.write("Upload DNA sequences to detect, analyze, and prioritize mutations using AI-like scoring.")

# -------------------- FILE UPLOAD --------------------
ref_file = st.file_uploader("Upload Reference DNA", type=["txt", "fasta"])
sample_file = st.file_uploader("Upload Sample DNA", type=["txt", "fasta"])

# -------------------- FUNCTIONS --------------------
def clean(seq):
    return ''.join([i for i in seq.upper() if i in "ATGC"])

def compare_sequences(ref, sample):
    mutations = []
    min_len = min(len(ref), len(sample))

    for i in range(min_len):
        if ref[i] != sample[i]:
            mutations.append({
                "position": i,
                "ref": ref[i],
                "sample": sample[i],
                "type": "Substitution"
            })

    if len(ref) > len(sample):
        mutations.append({"position": min_len, "type": "Deletion"})
    elif len(sample) > len(ref):
        mutations.append({"position": min_len, "type": "Insertion"})

    return mutations

def score_mutation(m):
    score = 0

    # Type weight
    if m["type"] == "Insertion" or m["type"] == "Deletion":
        score += 50
    else:
        score += 20

    # GC disruption
    ref = m.get("ref", "")
    sample = m.get("sample", "")
    if ref in "GC" and sample in "AT":
        score += 30

    # Position importance (middle region higher weight)
    score += int(30 * np.sin(m["position"] / 50))

    return min(score, 100)

def interpret_score(score):
    if score > 70:
        return "High Risk Mutation"
    elif score > 40:
        return "Moderate Impact"
    else:
        return "Low Impact"

# -------------------- MAIN LOGIC --------------------
if ref_file and sample_file:

    ref_seq = clean(ref_file.read().decode())
    sample_seq = clean(sample_file.read().decode())

    st.subheader("Sequence Info")
    col1, col2 = st.columns(2)
    col1.write(f"Reference Length: {len(ref_seq)}")
    col2.write(f"Sample Length: {len(sample_seq)}")

    mutations = compare_sequences(ref_seq, sample_seq)

    if len(mutations) == 0:
        st.success("No mutations detected!")
    else:
        st.subheader("Top Mutations (Ranked)")

        scored = []
        for m in mutations:
            s = score_mutation(m)
            m["score"] = s
            m["impact"] = interpret_score(s)
            scored.append(m)

        # Sort by score
        scored = sorted(scored, key=lambda x: x["score"], reverse=True)

        # Display top mutations
        for m in scored[:10]:
            color = "🔴" if m["score"] > 70 else "🟡" if m["score"] > 40 else "🟢"
            st.write(f"{color} Position {m['position']} | {m.get('ref','-')} → {m.get('sample','-')} | {m['type']} | Score: {m['score']} | {m['impact']}")

        st.success(f"Total Mutations Found: {len(mutations)}")

        # -------------------- ATTENTION MAP --------------------
        st.subheader("Mutation Attention Map")

        scores = [m["score"] for m in scored]

        fig1, ax1 = plt.subplots()
        ax1.plot(scores)
        ax1.set_title("Mutation Importance Across Genome")
        ax1.set_xlabel("Mutation Index")
        ax1.set_ylabel("Score")

        st.pyplot(fig1)

        # -------------------- GENOME SEGMENT VISUAL --------------------
        st.subheader("Genome GC Content Map")

        chunk_size = 50
        chunks = [ref_seq[i:i+chunk_size] for i in range(0, len(ref_seq), chunk_size)]

        gc_values = []
        for chunk in chunks:
            g = chunk.count('G')
            c = chunk.count('C')
            gc = (g + c) / len(chunk) if len(chunk) > 0 else 0
            gc_values.append(gc)

        fig2, ax2 = plt.subplots(figsize=(10, 2))
        ax2.bar(range(len(gc_values)), gc_values)
        ax2.set_title("GC Content Variation")
        ax2.set_xlabel("Genome Segments")
        ax2.set_ylabel("GC Ratio")

        st.pyplot(fig2)

        # -------------------- DOWNLOAD REPORT --------------------
        st.subheader("Download Mutation Report")

        df = pd.DataFrame(scored)
        csv = df.to_csv(index=False)

        st.download_button(
            label="Download CSV Report",
            data=csv,
            file_name="mutation_report.csv",
            mime="text/csv"
        )

else:
    st.info("Please upload both Reference and Sample DNA files to begin analysis.")